# Solar Filament Segmentation — YOLO instance segmentation

**Francesco Peluso** (`IE22700088`, f.peluso29@studenti.unisa.it)
Machine Learning A.Y. 2025/26 - MSc Computer Engineering, DIEM, University of Salerno
[filament-segmentation-2026](https://www.kaggle.com/competitions/filament-segmentation-2026)

> **Why this notebook exists.** Every experiment so far derives *instances* from a
> pixel-wise U-Net: connected components of a thresholded probability map, plus a
> confidence gate. The whole study concluded that segmentation quality (SQ) is capped
> by annotator disagreement and that every point of PQ goes through *recognition*
> (RQ): finding the right number of filaments. A detector-based instance segmenter
> has what that pipeline lacks, an **objectness head trained per instance** — and the
> leaderboard leader (public 0.53) uses exactly one, a YOLOv8-seg at 1792 px, with no
> ensemble and no post-processing. This notebook ports that idea into the project's
> protocol: same split, same official metric, same error analysis, operating point
> (`conf`, NMS `iou`, `max_det`) searched on validation instead of guessed.

Copied from the unconstrained U-Net notebook: the environment, the COCO loading and
the split, the official metric and the error-analysis cells are reused verbatim.
The memory budget of the assignment is not a constraint here.


## 1. Environment

The same notebook has to run on Colab (NVIDIA), on a MacBook (Apple Silicon) and on a
plain CPU. Instead of checking version numbers, the settings that actually differ
between backends — autocast dtype, gradient scaler, memory format, dataloader workers —
are probed once by just trying them.

In [ ]:
import subprocess
import sys
from pathlib import Path

ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocotools", "ultralytics"],
                   check=False)


def locate_project() -> Path:
    """Find the project root from wherever this notebook was opened."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "dataset").is_dir() or (candidate / "experiments").is_dir():
            return candidate
    for candidate in (Path("/content/MachineLearning-Project"),
                      Path("/kaggle/working/MachineLearning-Project")):
        if candidate.is_dir():
            return candidate
    return Path.cwd()


PROJECT_ROOT = locate_project()

# The competition zip's own layout nests everything one level deeper
# (dataset/MAGFiLO_1.0_Kaggle_2026/train/...) than a plain "dataset/train/..." tree
# does. Both are accepted below, whichever a local copy or a kagglehub download hands
# back, so no extra path-fixing cell is ever needed.
def _has_dataset(root: Path) -> bool:
    return (root / "train").is_dir() or (root / "MAGFiLO_1.0_Kaggle_2026").is_dir()


DATASET_DIR = PROJECT_ROOT / "dataset"

if ON_COLAB and not _has_dataset(DATASET_DIR):
    # Nothing local yet (no Drive mount, no manual unzip): pull the competition data
    # straight from Kaggle. This needs Kaggle credentials available in the session
    # (Colab Secrets KAGGLE_USERNAME/KAGGLE_KEY, a kaggle.json, or kagglehub's own
    # interactive login) and the competition rules accepted on kaggle.com - if either
    # is missing, kagglehub raises its own clear error here rather than failing later
    # with a bare FileNotFoundError deep inside data loading.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=False)
    import kagglehub
    DATASET_DIR = Path(kagglehub.competition_download("filament-segmentation-2026"))
    print(f"downloaded competition data to {DATASET_DIR}")

if not (DATASET_DIR / "train").is_dir() and (DATASET_DIR / "MAGFiLO_1.0_Kaggle_2026").is_dir():
    DATASET_DIR = DATASET_DIR / "MAGFiLO_1.0_Kaggle_2026"
TRAIN_IMAGES = DATASET_DIR / "train" / "train_images"
ANNOTATIONS = DATASET_DIR / "train" / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
TEST_IMAGES = DATASET_DIR / "test" / "test_images"

print(f"project : {PROJECT_ROOT}")
print(f"dataset : {DATASET_DIR}  (exists: {DATASET_DIR.is_dir()})")
if not ANNOTATIONS.is_file():
    listing = sorted(p.name for p in DATASET_DIR.iterdir()) if DATASET_DIR.is_dir() else []
    print(f"WARNING: annotations not found at {ANNOTATIONS}")
    print(f"  contents of {DATASET_DIR}: {listing}")

In [ ]:
import json
import os
import platform
import random
import sys
import time
from contextlib import nullcontext

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw
from scipy import ndimage
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 22


def seed_everything(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class Runtime:
    """Everything that depends on the accelerator we happen to be running on
    (CUDA on Colab, MPS on a MacBook, or plain CPU), probed once at start-up."""

    def __init__(self, num_workers=None):
        if torch.cuda.is_available():
            self.device = torch.device("cuda")
            torch.backends.cudnn.benchmark = True
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True
        elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
            self.device = torch.device("mps")
            torch.set_float32_matmul_precision("high")
        else:
            self.device = torch.device("cpu")

        self.amp_dtype, self.needs_scaler = self._probe_autocast()
        self.channels_last = self.device.type == "cuda"
        self.pin_memory = self.device.type == "cuda"
        # With the uint8 cache in place one process already feeds ~160 samples/s,
        # more than any of these GPUs can consume, so workers only pay off on
        # CUDA hosts where fork is cheap.
        cpus = os.cpu_count() or 2
        self.num_workers = num_workers if num_workers is not None else (
            min(4, max(0, cpus - 2)) if self.device.type == "cuda" else 0)
        self._peak = 0

    def _probe_autocast(self):
        if self.device.type == "cpu":
            return None, False
        for dtype in (torch.bfloat16, torch.float16):
            try:
                with torch.autocast(device_type=self.device.type, dtype=dtype):
                    probe = torch.ones(8, 8, device=self.device)
                    if float((probe @ probe).float().sum()) != 512.0:
                        continue
            except Exception:
                continue
            return dtype, dtype is torch.float16 and self.device.type == "cuda"
        return None, False

    def autocast(self):
        return (nullcontext() if self.amp_dtype is None
                else torch.autocast(device_type=self.device.type, dtype=self.amp_dtype))

    def grad_scaler(self):
        return torch.amp.GradScaler(self.device.type, enabled=self.needs_scaler)

    def prepare(self, obj):
        obj = obj.to(self.device, non_blocking=self.pin_memory)
        if self.channels_last and (not torch.is_tensor(obj) or obj.dim() == 4):
            obj = obj.to(memory_format=torch.channels_last)
        return obj

    def loader_kwargs(self, batch_size):
        kwargs = {"batch_size": batch_size, "num_workers": self.num_workers,
                  "pin_memory": self.pin_memory}
        if self.num_workers:
            # Workers must be recreated at every epoch. FilamentDataset.set_epoch()
            # changes the deterministic crop/augmentation seed in the main process;
            # persistent workers would keep their private epoch-0 dataset copy and
            # therefore repeat the same crop, flips and photometric transform forever.
            kwargs.update(persistent_workers=False, prefetch_factor=4)
        return kwargs

    # Peak memory: the assignment gives a bonus below 5 GB training / 4 GB test.
    # CUDA tracks the peak itself; MPS has no peak counter and its pool figure
    # over-reports, so live allocation is sampled at the point a step peaks.
    def reset_peak_memory(self):
        self._peak = 0
        if self.device.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.empty_cache()
        elif self.device.type == "mps":
            torch.mps.empty_cache()

    def note_memory(self):
        if self.device.type == "mps":
            self._peak = max(self._peak, torch.mps.current_allocated_memory())

    def peak_memory_gb(self):
        """Peak of live tensors: comparable across backends, the figure to quote."""
        if self.device.type == "cuda":
            return torch.cuda.max_memory_allocated() / 1024 ** 3
        if self.device.type == "mps":
            return (self._peak or torch.mps.driver_allocated_memory()) / 1024 ** 3
        return float("nan")

    def reserved_memory_gb(self):
        """What the allocator holds, cache included: a conservative upper bound.

        In inference the live figure under-reports, because nothing keeps the
        activations alive long enough to be sampled between steps; this one does
        not have that problem.
        """
        if self.device.type == "cuda":
            return torch.cuda.max_memory_reserved() / 1024 ** 3
        if self.device.type == "mps":
            return torch.mps.driver_allocated_memory() / 1024 ** 3
        return float("nan")

    def summary(self):
        name = (torch.cuda.get_device_name(0) if self.device.type == "cuda"
                else f"Apple Silicon GPU ({platform.machine()})" if self.device.type == "mps"
                else platform.processor() or "CPU")
        dtype = str(self.amp_dtype).replace("torch.", "") if self.amp_dtype else "disabled"
        return (f"device       : {self.device} ({name})\n"
                f"autocast     : {dtype}{' + GradScaler' if self.needs_scaler else ''}\n"
                f"channels_last: {self.channels_last} | dataloader workers: {self.num_workers}\n"
                f"torch {torch.__version__} | python {sys.version.split()[0]}")

    def info(self):
        return {"device": str(self.device), "autocast": str(self.amp_dtype),
                "channels_last": self.channels_last, "num_workers": self.num_workers,
                "torch": torch.__version__, "python": sys.version.split()[0],
                "platform": platform.platform()}


seed_everything()
runtime = Runtime()
print(runtime.summary())

## 2. Data and the split

The annotations are in COCO format, with a twist: each entry of `images` is **one
annotator's view** of an observation, identified as `<annotator>-<file stem>`. The same
JPEG can appear two or three times with different polygons, because the annotators
disagree on where a filament starts and ends. Those disagreements are kept: the official
metric scores a prediction against every annotator separately.

The train/validation/test split is done on **file names**, never on annotator records:
two annotators of the same image describe the same pixels, so splitting on records would
leak validation pixels into training and inflate every score.

In [ ]:
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass

NATIVE = 2048


@dataclass
class Sample:
    image_id: str          # "<annotator>-<file stem>", the competition's own identifier
    file_name: str
    image_path: Path
    width: int
    height: int
    polygons: list


def load_coco(annotations_path, images_dir):
    """Each entry of `images` is one annotator's view of one observation."""
    with Path(annotations_path).open(encoding="utf-8") as handle:
        coco = json.load(handle)

    polygons_by_image = defaultdict(list)
    for annotation in coco["annotations"]:
        for polygon in annotation.get("segmentation", []):
            if len(polygon) >= 6:              # a polygon needs at least three points
                polygons_by_image[annotation["image_id"]].append(polygon)

    samples = [Sample(info["id"], info["file_name"], Path(images_dir) / info["file_name"],
                      info["width"], info["height"], polygons_by_image[info["id"]])
               for info in coco["images"]]
    missing = [s.image_path for s in samples if not s.image_path.is_file()]
    if missing:
        raise FileNotFoundError(f"{len(missing)} images missing, first: {missing[0]}")
    return samples


def split_by_file(samples, train_fraction=0.70, validation_fraction=0.15, seed=SEED):
    """Split on file names, never on records.

    Two annotators of the same observation describe the same pixels; splitting on
    records would put one copy in training and another in validation and inflate
    every score without improving the model at all.
    """
    file_names = sorted({s.file_name for s in samples})
    random.Random(seed).shuffle(file_names)
    train_end = int(len(file_names) * train_fraction)
    validation_end = train_end + int(len(file_names) * validation_fraction)
    files = {"train": set(file_names[:train_end]),
             "validation": set(file_names[train_end:validation_end]),
             "test": set(file_names[validation_end:])}
    by_split = {name: [s for s in samples if s.file_name in group]
                for name, group in files.items()}
    return by_split, files


def rasterize(width, height, polygons, size=None):
    mask = Image.new("L", (width, height), 0)
    drawer = ImageDraw.Draw(mask)
    for polygon in polygons:
        drawer.polygon([(polygon[i], polygon[i + 1]) for i in range(0, len(polygon), 2)], fill=1)
    if size is not None and (width, height) != (size, size):
        mask = mask.resize((size, size), resample=Image.Resampling.NEAREST)
    return np.asarray(mask, dtype=np.uint8)


def rasterize_instances(width, height, polygons, size=None):
    """One label per polygon: the GT instances the metric counts, for the loss.

    The dataset has at most 26 polygons per annotator record, so uint8 is enough.
    Overlapping polygons are rare and the later one wins; a loss term does not
    need the disputed sliver, the metric keeps scoring both instances in full.
    """
    mask = Image.new("L", (width, height), 0)
    drawer = ImageDraw.Draw(mask)
    for value, polygon in enumerate(polygons, start=1):
        drawer.polygon([(polygon[i], polygon[i + 1]) for i in range(0, len(polygon), 2)],
                       fill=value)
    if size is not None and (width, height) != (size, size):
        mask = mask.resize((size, size), resample=Image.Resampling.NEAREST)
    return np.asarray(mask, dtype=np.uint8)


def load_image(path, size=None):
    with Image.open(path) as handle:
        image = handle.convert("L")
        if size is not None and image.size != (size, size):
            image = image.resize((size, size), resample=Image.Resampling.BILINEAR)
        return np.asarray(image, dtype=np.uint8)


class SampleCache:
    """Images and masks decoded once into uint8 memory-mapped files on disk,
    shared across runs. Masks are bit-packed (8 pixels per byte)."""

    def __init__(self, samples, size, cache_dir, workers=None, soft=False,
                 instances=False):
        self.dir = Path(cache_dir)
        self.dir.mkdir(parents=True, exist_ok=True)
        self.size = size
        file_names = sorted({s.file_name for s in samples})
        self.file_row = {name: i for i, name in enumerate(file_names)}
        self.record_row = {s.image_id: i for i, s in enumerate(samples)}
        self._images = self._masks = self._soft = self._labels = None
        self.has_soft = bool(soft)
        self.has_instances = bool(instances)

        index_path = self.dir / f"index_{size}.json"
        images_path = self.dir / f"images_{size}.u8"
        masks_path = self.dir / f"masks_{size}.u8"
        soft_path = self.dir / f"soft_{size}.u8"
        labels_path = self.dir / f"labels_{size}.u8"
        index = {"size": size, "files": file_names, "records": [s.image_id for s in samples],
                 "soft": self.has_soft}
        if self.has_instances:
            # only stamped when requested, so caches built before this flag existed
            # stay valid for every run that does not need instance labels
            index["instances"] = True
        stored = json.loads(index_path.read_text()) if index_path.is_file() else None
        if stored is not None and not self.has_instances:
            stored.pop("instances", None)
        if (stored == index and images_path.is_file() and masks_path.is_file()
                and (soft_path.is_file() or not self.has_soft)
                and (labels_path.is_file() or not self.has_instances)):
            return

        images = np.lib.format.open_memmap(images_path, mode="w+", dtype=np.uint8,
                                           shape=(len(file_names), size, size))
        masks = np.lib.format.open_memmap(masks_path, mode="w+", dtype=np.uint8,
                                          shape=(len(samples), size * size // 8))
        soft = (np.lib.format.open_memmap(soft_path, mode="w+", dtype=np.uint8,
                                          shape=(len(file_names), size, size))
                if self.has_soft else None)
        labels = (np.lib.format.open_memmap(labels_path, mode="w+", dtype=np.uint8,
                                            shape=(len(samples), size, size))
                  if self.has_instances else None)
        path_by_file = {s.file_name: s.image_path for s in samples}
        records_by_file = defaultdict(list)
        for sample in samples:
            records_by_file[sample.file_name].append(sample)

        # Threads, not processes: JPEG decoding and polygon rasterisation release
        # the GIL, and unlike `spawn` this behaves the same in a script, a
        # notebook, Colab and Kaggle.
        def do_image(item):
            i, name = item
            images[i] = load_image(path_by_file[name], size)

        def do_mask(item):
            i, sample = item
            masks[i] = np.packbits(rasterize(sample.width, sample.height, sample.polygons, size))

        def do_labels(item):
            i, sample = item
            labels[i] = rasterize_instances(sample.width, sample.height,
                                            sample.polygons, size)

        def do_soft(item):
            """The fraction of this observation's annotators covering each pixel.

            Averaged at native resolution and only then resized, with an antialiased
            filter: a boundary pixel half of the panel included arrives as 0.5 instead of
            being rounded to whichever annotator the nearest-neighbour grid happened to
            land on.
            """
            i, name = item
            records = records_by_file[name]
            total = np.zeros((NATIVE, NATIVE), dtype=np.float32)
            for sample in records:
                total += rasterize(sample.width, sample.height, sample.polygons)
            plane = Image.fromarray((total * (255.0 / len(records))).astype(np.uint8))
            if size != NATIVE:
                plane = plane.resize((size, size), resample=Image.Resampling.BILINEAR)
            soft[i] = np.asarray(plane, dtype=np.uint8)

        with ThreadPoolExecutor(max_workers=workers or min(8, os.cpu_count() or 4)) as pool:
            list(tqdm(pool.map(do_image, enumerate(file_names)), total=len(file_names),
                      desc=f"cache images @{size}", leave=False))
            list(tqdm(pool.map(do_mask, enumerate(samples)), total=len(samples),
                      desc=f"cache masks @{size}", leave=False))
            if self.has_soft:
                list(tqdm(pool.map(do_soft, enumerate(file_names)), total=len(file_names),
                          desc=f"cache soft targets @{size}", leave=False))
            if self.has_instances:
                list(tqdm(pool.map(do_labels, enumerate(samples)), total=len(samples),
                          desc=f"cache instance labels @{size}", leave=False))
        images.flush(); masks.flush()
        if soft is not None:
            soft.flush()
        if labels is not None:
            labels.flush()
        del images, masks, soft, labels
        index_path.write_text(json.dumps(index), encoding="utf-8")

    def _open(self):
        if self._images is None:
            self._images = np.load(self.dir / f"images_{self.size}.u8", mmap_mode="r")
            self._masks = np.load(self.dir / f"masks_{self.size}.u8", mmap_mode="r")
            if self.has_soft:
                self._soft = np.load(self.dir / f"soft_{self.size}.u8", mmap_mode="r")
            if self.has_instances:
                self._labels = np.load(self.dir / f"labels_{self.size}.u8", mmap_mode="r")

    def image(self, file_name):
        self._open()
        return np.asarray(self._images[self.file_row[file_name]])

    def mask(self, image_id):
        self._open()
        return np.unpackbits(self._masks[self.record_row[image_id]]).reshape(self.size, self.size)

    def soft(self, file_name):
        """Consensus target for an observation, as 0-255 = fraction of annotators."""
        self._open()
        return np.asarray(self._soft[self.file_row[file_name]])

    def labels(self, image_id):
        """Instance labels of one annotator record: 0 background, 1..k per polygon."""
        self._open()
        return np.asarray(self._labels[self.record_row[image_id]])

    def nbytes(self):
        keys = ["images", "masks"]
        if self.has_soft:
            keys.append("soft")
        if self.has_instances:
            keys.append("labels")
        return sum((self.dir / f"{k}_{self.size}.u8").stat().st_size for k in keys)


class SegmentationTransform:
    """Label-preserving augmentation applied to image and mask together.

    Filaments have no canonical orientation, so the eight symmetries of the
    square are free data. Photometric jitter exists because H-alpha contrast
    changes between observations; `scale_jitter` makes the network tolerant of a
    test-time scale different from the training one.
    """

    def __init__(self, augment=False, photometric=True, scale_jitter=None, soft=False,
                 crop=None):
        self.augment, self.photometric, self.scale_jitter = augment, photometric, scale_jitter
        self.soft = soft       # a soft target arrives as 0-255 and must interpolate, not snap
        self.crop = crop       # train on a window of the native frame, not a shrunken copy

    def __call__(self, image, mask, rng):
        if self.augment and self.crop and self.crop < image.shape[0]:
            image, mask = self._window(image, mask, rng)
        if self.augment and self.scale_jitter is not None:
            image, mask = self._rescale(image, mask, rng)
        if self.augment:
            if rng.random() < 0.5:
                image, mask = image[:, ::-1], mask[:, ::-1]
            if rng.random() < 0.5:
                image, mask = image[::-1], mask[::-1]
            turns = rng.randint(0, 3)
            if turns:
                image, mask = np.rot90(image, turns), np.rot90(mask, turns)

        image = np.ascontiguousarray(image, dtype=np.float32) / 255.0
        if self.augment and self.photometric:
            image *= 1.0 + rng.uniform(-0.20, 0.20)
            mean = float(image.mean())
            image = (image - mean) * (1.0 + rng.uniform(-0.20, 0.20)) + mean
            if rng.random() < 0.30:
                # Use the sample/epoch RNG here too: global NumPy state depends on
                # worker scheduling and makes otherwise deterministic runs diverge.
                noise = np.random.default_rng(rng.randrange(2 ** 32))
                image += noise.standard_normal(image.shape, dtype=np.float32) * 0.02
            np.clip(image, 0.0, 1.0, out=image)

        mask = np.ascontiguousarray(mask, dtype=np.float32)
        if self.soft:
            mask /= 255.0
        return torch.from_numpy(image).unsqueeze(0), torch.from_numpy(mask).unsqueeze(0)

    def _window(self, image, mask, rng):
        """A random crop x crop window of the full-resolution frame.

        Downscaling the image to fit memory destroys exactly the small filaments
        the model already struggles with; a window keeps every pixel at its
        annotated scale and gives up field of view instead, which is cheap
        because filaments are local structures.
        """
        size = image.shape[0]
        top, left = rng.randint(0, size - self.crop), rng.randint(0, size - self.crop)
        return (image[top:top + self.crop, left:left + self.crop],
                mask[top:top + self.crop, left:left + self.crop])

    def _rescale(self, image, mask, rng):
        size = image.shape[0]
        scaled = max(64, int(round(size * rng.uniform(*self.scale_jitter))))
        if scaled == size:
            return image, mask
        image = np.asarray(Image.fromarray(image).resize((scaled, scaled), Image.Resampling.BILINEAR))
        resample = Image.Resampling.BILINEAR if self.soft else Image.Resampling.NEAREST
        mask = np.asarray(Image.fromarray(mask).resize((scaled, scaled), resample))
        if scaled > size:
            top, left = rng.randint(0, scaled - size), rng.randint(0, scaled - size)
            return image[top:top + size, left:left + size], mask[top:top + size, left:left + size]
        pad = size - scaled
        top, left = rng.randint(0, pad), rng.randint(0, pad)
        spec = ((top, pad - top), (left, pad - left))
        return np.pad(image, spec), np.pad(mask, spec)


class FilamentDataset(Dataset):
    def __init__(self, samples, cache, transform, seed=SEED, instance_targets=False):
        self.samples, self.cache, self.transform, self.seed = samples, cache, transform, seed
        # With instance targets the mask tensor carries the label image (0..k per
        # polygon) instead of 0/1. Geometric augmentation is label-safe as long as
        # nothing interpolates mask values; scale jitter does, so it is excluded.
        self.instance_targets = instance_targets
        if instance_targets:
            assert transform.scale_jitter is None, \
                "scale jitter interpolates instance labels"
            assert not transform.soft, "soft and instance targets are exclusive"
        self._epoch = 0

    def set_epoch(self, epoch):
        self._epoch = epoch

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        rng = random.Random((self.seed, self._epoch, index).__hash__())
        target = (self.cache.labels(sample.image_id) if self.instance_targets
                  else self.cache.mask(sample.image_id))
        image, mask = self.transform(self.cache.image(sample.file_name), target, rng)
        return {"image": image, "mask": mask,
                "file_name": sample.file_name, "image_id": sample.image_id}


class SoftConsensusDataset(Dataset):
    """One sample per observation, with the annotators averaged into a soft target.

    `FilamentDataset` emits one sample per annotator, so an observation labelled by three
    people enters every epoch three times with three hard masks that contradict each
    other. Measured on this dataset that contradiction is not a detail: of the pixels at
    least one annotator calls filament, only 40% are called filament by all of them, and
    on the three-annotator files only 30%. A hard 0/1 target gives the network no way to
    say a pixel is genuinely borderline, so it can only average the conflict out across
    epochs, as gradient noise.

    Here the target is the fraction of annotators covering each pixel. The disagreement
    becomes the label instead of noise on top of it, and the number the network learns to
    output means something concrete - how much of the panel would include this pixel -
    which is also what makes a probability threshold a meaningful thing to search over.

    The validation set deliberately keeps `FilamentDataset` and its per-annotator hard
    masks, so every score stays comparable with the earlier experiments and the official
    metric is untouched.
    """

    def __init__(self, samples, cache, transform, seed=SEED):
        self.files = sorted({s.file_name for s in samples})
        self.annotators = Counter(s.file_name for s in samples)
        self.cache, self.transform, self.seed = cache, transform, seed
        self._epoch = 0

    def set_epoch(self, epoch):
        self._epoch = epoch

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        file_name = self.files[index]
        rng = random.Random((self.seed, self._epoch, index).__hash__())
        image, mask = self.transform(self.cache.image(file_name),
                                     self.cache.soft(file_name), rng)
        return {"image": image, "mask": mask,
                "file_name": file_name, "image_id": Path(file_name).stem}


In [ ]:
samples = load_coco(ANNOTATIONS, TRAIN_IMAGES)
by_split, files_by_split = split_by_file(samples)

assert files_by_split["train"].isdisjoint(files_by_split["validation"])
assert files_by_split["train"].isdisjoint(files_by_split["test"])
assert files_by_split["validation"].isdisjoint(files_by_split["test"])

records_per_file = Counter(s.file_name for s in samples)
print(f"unique JPEG files : {len(records_per_file)}")
print(f"COCO records      : {len(samples)}")
print(f"filament polygons : {sum(len(s.polygons) for s in samples)}")
print(f"annotators/file   : {dict(sorted(Counter(records_per_file.values()).items()))}\n")
for name in ("train", "validation", "test"):
    print(f"{name:11s}: {len(files_by_split[name]):3d} files, {len(by_split[name]):3d} records, "
          f"{sum(len(s.polygons) for s in by_split[name]):4d} polygons")

# The assignment asks for the split protocol as a CSV of sample names.
import csv
split_csv = PROJECT_ROOT / "splits" / f"split_seed{SEED}.csv"
split_csv.parent.mkdir(parents=True, exist_ok=True)
with split_csv.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["file_name", "split"])
    for name in ("train", "validation", "test"):
        for file_name in sorted(files_by_split[name]):
            writer.writerow([file_name, name])
print(f"\nwritten: {split_csv.relative_to(PROJECT_ROOT)}")

## 3. Configuration

**The only cell to edit between experiments.**

In [ ]:
EXPERIMENT_ID = "Y4"

RUN_TRAINING = True            # False reuses experiments/<ID>/checkpoints/best_model.pt
RUN_FINAL_TEST = False
TRAIN_ON_TEST_SPLIT = True     # as every run from experiment 07 on

# --- detector ------------------------------------------------------------------
# ultralytics weights: "yolov8n-seg.pt" (smoke) | "yolov8m-seg.pt" | "yolov8l-seg.pt"
# | "yolov8x-seg.pt" (the leaderboard leader's). Memory at IMG_SIZE 1792, batch 4:
# m ~9 GB, x ~14 GB (Colab T4 fits m/l; x wants an L4/A100 or batch 2).
YOLO_MODEL = "yolov8x-seg.pt"
IMG_SIZE = 1792                # multiple of 32; 1792 = the leader's choice, near native
BATCH = -1                     # autobatch: ultralytics sizes it to ~60% of VRAM
EPOCHS = 100
PATIENCE = 30                  # ultralytics early stopping on its own fitness
# How the 2-3 annotators of a file become one training target:
#   "all"    every record is its own training image (conflicting objectness targets;
#            capped the detector's recall at ~0.60 in Y1/Y2);
#   "first"  one annotator per file (clean but discards the others' filaments, Y3);
#   "union"  every filament ANY annotator marked, duplicates of the same filament
#            removed at mask IoU > 0.5 keeping the largest. The measured marginal
#            inclusion rule says a filament is worth predicting when at least a
#            third of annotators would mark it - which the union guarantees by
#            construction. This is the recall-maximising target set.
MULTI_ANNOTATOR = "union"      # "all" | "first" | "union"
# Augmentation: filaments have no orientation (both flips), no rotation, mild scale.
# Y3 replicates the leaderboard leader as faithfully as the evidence allows:
# yolov8x, one annotator per file, ultralytics DEFAULT augmentation. The single
# deviation is flipud (filaments have no vertical orientation either). Y1/Y2
# falsified undertraining as the explanation of the 0.42 vs 0.53 gap; if Y3
# lands, the ablation runs backwards from here (mosaic, "all", model size).
AUG = {"flipud": 0.5}

# --- operating point, searched on validation against the official PQ ----------
# Prediction runs once per NMS iou at a permissive confidence; conf and max_det are
# then applied afterwards (exact: NMS only ever suppresses lower-confidence boxes).
SEARCH_IOU = (0.0, 0.3, 0.5, 0.7)
SEARCH_CONF = (0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.60)
SEARCH_MAX_DET = (50, 100, 300)
PREDICT_CONF = 0.05            # floor for the stored candidates
RETINA_MASKS = True            # masks at image resolution instead of the proto grid

SEED = 22
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / EXPERIMENT_ID
METRICS_DIR, PLOTS_DIR = EXPERIMENT_DIR / "metrics", EXPERIMENT_DIR / "plots"
CHECKPOINTS_DIR = EXPERIMENT_DIR / "checkpoints"
for directory in (METRICS_DIR, PLOTS_DIR, CHECKPOINTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
BEST_CHECKPOINT = CHECKPOINTS_DIR / "best_model.pt"
YOLO_DATA_DIR = PROJECT_ROOT / ".cache" / f"yolo_{MULTI_ANNOTATOR}"


def save_json(data, path):
    Path(path).write_text(json.dumps(data, indent=2, default=float), encoding="utf-8")


print(f"artifacts -> {EXPERIMENT_DIR.relative_to(PROJECT_ROOT)}")
print(f"{YOLO_MODEL} at {IMG_SIZE}px, batch {BATCH}, {EPOCHS} epochs | "
      f"{'training' if RUN_TRAINING else 'reusing checkpoint'}")

## 4. The dataset in YOLO format

One `.txt` per image with one line per filament: class `0` followed by the polygon
vertices normalised to `[0, 1]`. Images are symlinks into `dataset/`, named after the
annotator record so that a file annotated by three people appears three times with
three label files. The split is the project's seed-22 split by file name.


In [ ]:
import shutil

training_samples = by_split["train"] + (by_split["test"] if TRAIN_ON_TEST_SPLIT else [])
validation_samples = by_split["validation"]


def keep_one_per_file(records):
    seen, kept = set(), []
    for sample in records:
        if sample.file_name not in seen:
            seen.add(sample.file_name)
            kept.append(sample)
    return kept


if MULTI_ANNOTATOR == "first":
    training_samples = keep_one_per_file(training_samples)
    validation_samples = keep_one_per_file(validation_samples)


def merge_annotators(records):
    """One synthetic record per file: the union of every annotator's filaments,
    de-duplicated at mask IoU > 0.5 (largest polygon wins). IoU is computed on
    512x512 rasters, plenty for a duplicate test."""
    from collections import defaultdict
    by_file = defaultdict(list)
    for sample in records:
        by_file[sample.file_name].append(sample)
    merged = []
    for file_name, group in by_file.items():
        first = group[0]
        candidates = []
        for sample in group:
            for polygon in sample.polygons:
                raster = rasterize(sample.width, sample.height, [polygon], 512)
                candidates.append((int(raster.sum()), polygon,
                                   np.packbits(raster)))
        candidates.sort(key=lambda item: -item[0])
        kept, kept_bits = [], []
        for area, polygon, bits in candidates:
            mask = np.unpackbits(bits)
            duplicate = False
            for other_bits in kept_bits:
                other = np.unpackbits(other_bits)
                inter = int(np.logical_and(mask, other).sum())
                union = int(np.logical_or(mask, other).sum())
                if union and inter / union > 0.5:
                    duplicate = True
                    break
            if not duplicate:
                kept.append(polygon)
                kept_bits.append(bits)
        merged.append(Sample(Path(file_name).stem, file_name, first.image_path,
                             first.width, first.height, kept))
    return merged


if MULTI_ANNOTATOR == "union":
    training_samples = merge_annotators(training_samples)
    validation_samples = merge_annotators(validation_samples)


def export_split(records, name):
    images = YOLO_DATA_DIR / "images" / name
    labels = YOLO_DATA_DIR / "labels" / name
    for directory in (images, labels):
        if directory.exists():
            shutil.rmtree(directory)
        directory.mkdir(parents=True)
    for sample in records:
        link = images / f"{sample.image_id}.jpeg"
        try:
            link.symlink_to(sample.image_path.resolve())
        except OSError:
            shutil.copyfile(sample.image_path, link)
        lines = []
        for polygon in sample.polygons:
            xs, ys = polygon[0::2], polygon[1::2]
            coords = " ".join(f"{min(max(x / sample.width, 0.0), 1.0):.6f} "
                              f"{min(max(y / sample.height, 0.0), 1.0):.6f}"
                              for x, y in zip(xs, ys))
            lines.append(f"0 {coords}")
        (labels / f"{sample.image_id}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
    return len(records)


n_train = export_split(training_samples, "train")
n_val = export_split(validation_samples, "val")
data_yaml = YOLO_DATA_DIR / "data.yaml"
data_yaml.write_text(f"path: {YOLO_DATA_DIR}\ntrain: images/train\nval: images/val\n"
                     f"names:\n  0: filament\n")
print(f"YOLO dataset at {YOLO_DATA_DIR.relative_to(PROJECT_ROOT)}: "
      f"{n_train} training images, {n_val} validation images "
      f"({sum(len(s.polygons) for s in training_samples)} training instances)")

## 5. Training

`ultralytics` owns the loop: its loss already weighs every instance the same (box,
objectness and mask per instance), which is the property the pixel-wise experiments
could not obtain. Its `best.pt` is selected on its own fitness (mask mAP); the
official PQ then decides the operating point in the next section.


In [ ]:
from ultralytics import YOLO

device = (0 if torch.cuda.is_available() else
          "mps" if getattr(torch.backends, "mps", None) is not None
          and torch.backends.mps.is_available() else "cpu")
training_summary = None
if RUN_TRAINING:
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    yolo = YOLO(YOLO_MODEL)
    results = yolo.train(data=str(data_yaml), imgsz=IMG_SIZE, epochs=EPOCHS, batch=BATCH,
                         patience=PATIENCE, seed=SEED, deterministic=True, device=device,
                         project=str(EXPERIMENT_DIR / "yolo"), name="train",
                         exist_ok=True, plots=True, verbose=False, **AUG)
    run_dir = Path(results.save_dir)
    shutil.copyfile(run_dir / "weights" / "best.pt", BEST_CHECKPOINT)
    if (run_dir / "results.csv").exists():
        shutil.copyfile(run_dir / "results.csv", METRICS_DIR / "history.csv")
    peak = (torch.cuda.max_memory_allocated() / 1024 ** 3
            if torch.cuda.is_available() else float("nan"))
    training_summary = {"model": YOLO_MODEL, "image_size": IMG_SIZE, "batch": BATCH,
                        "epochs_requested": EPOCHS,
                        "total_minutes": (time.perf_counter() - started) / 60,
                        "peak_training_memory_gb": peak,
                        "run_dir": str(run_dir.relative_to(PROJECT_ROOT))}
    save_json(training_summary, METRICS_DIR / "training_summary.json")
    print(f"trained in {training_summary['total_minutes']:.1f} min, "
          f"peak memory {peak:.2f} GB -> {BEST_CHECKPOINT.relative_to(PROJECT_ROOT)}")
else:
    training_summary = json.loads((METRICS_DIR / "training_summary.json").read_text())
    print(f"reusing {BEST_CHECKPOINT.relative_to(PROJECT_ROOT)}")
yolo = YOLO(str(BEST_CHECKPOINT))

## 7. Metrics

`official_pq` is the competition's Panoptic Quality, re-implemented from the organisers'
notebook and verified against it. Three details make it different from a textbook PQ,
and all three change what a good prediction looks like:

- ground truth is **one instance per COCO annotation**, not connected components of the
  union of all annotators' polygons;
- **each annotator is a separate evaluation entry**, so one prediction is scored two or
  three times against people who disagree with each other;
- TP, FP and FN are summed **globally over the whole split**, not averaged per image.

The organisers' code builds dense `(n, 2048, 2048)` tensors and multiplies them. Here
the ground truth is kept as pixel indices and the prediction as a label image, which
gives identical numbers much faster.

In [ ]:
from pycocotools import mask as mask_util


def postprocess(mask, min_size=0, close=0):
    """Close small gaps, then drop components below `min_size` pixels.

    Both steps target Panoptic Quality rather than Dice. Closing reconnects a
    filament the network split in two, converting two false positives and one
    false negative into one true positive; removing tiny components deletes the
    specks that each count as a separate false-positive instance while
    contributing almost nothing to the pixel overlap.
    """
    mask = np.asarray(mask, dtype=bool)
    if close:
        mask = ndimage.binary_closing(mask, iterations=close)
    if min_size:
        labels, count = ndimage.label(mask)
        if count:
            areas = np.bincount(labels.ravel(), minlength=count + 1)
            keep = areas >= min_size
            keep[0] = False
            mask = keep[labels]
    return mask


# --------------------------------------------------------------------------- #
# The competition's Panoptic Quality
# --------------------------------------------------------------------------- #
# Reimplemented from the organisers' self-evaluation notebook. Three details
# make it different from a naive PQ, and all three change what a good prediction
# looks like:
#
#   * ground truth is one instance per COCO annotation, not connected components
#     of the union of every annotator's polygons;
#   * each annotator of an image is a separate evaluation entry, so the same
#     prediction is scored two or three times against people who disagree with
#     each other;
#   * TP/FP/FN are accumulated globally over the whole split, not averaged per
#     image.
#
# The organisers' version builds dense (n, 2048, 2048) tensors and multiplies
# them. Here the ground truth stays as pixel indices and the prediction as a
# label image, which gives identical numbers at a fraction of the cost.

def build_ground_truth(samples, size=NATIVE):
    """One entry per annotator-image, one index array per annotated filament."""
    entries = []
    for sample in tqdm(list(samples), desc="official ground truth", leave=False):
        instances = []
        for polygon in sample.polygons:
            rles = mask_util.frPyObjects([polygon], size, size)
            rle = mask_util.merge(rles) if isinstance(rles, list) else rles
            index = np.flatnonzero(mask_util.decode(rle).ravel())
            if index.size:
                instances.append(index)
        entries.append({"annotator_image": sample.image_id,
                        "stem": Path(sample.file_name).stem,
                        "instances": instances,
                        # union of the instances, precomputed: mean_dice needs it
                        # for every point of the grid and it never changes
                        "union": (np.unique(np.concatenate(instances)) if instances
                                  else np.zeros(0, dtype=np.int64))})
    return entries


def official_pq(ground_truth, predictions, iou_threshold=0.5):
    """`predictions` maps a file stem to a (flat label image, count, areas) triple,
    as produced by `predictions_at`."""
    tp_iou, false_positive, false_negative = [], 0, 0
    for entry in ground_truth:
        labels, n_pred, pred_areas = predictions.get(
            entry["stem"], (None, 0, np.zeros(0, dtype=np.int64)))
        n_gt = len(entry["instances"])
        if n_gt == 0:
            false_positive += n_pred
            continue
        if n_pred == 0:
            false_negative += n_gt
            continue
        iou = np.zeros((n_gt, n_pred))
        for i, index in enumerate(entry["instances"]):
            counts = np.bincount(labels[index], minlength=n_pred + 1)[1:]
            union = index.size + pred_areas - counts
            np.divide(counts, np.maximum(union, 1), out=iou[i], where=union > 0)
        hit = iou > iou_threshold
        tp_iou.extend(iou[hit].tolist())
        false_positive += int((hit.sum(axis=0) == 0).sum())
        false_negative += int((hit.sum(axis=1) == 0).sum())
    denominator = len(tp_iou) + 0.5 * false_positive + 0.5 * false_negative
    return {"pq": (sum(tp_iou) / denominator) if denominator else 0.0,
            "tp": len(tp_iou), "fp": false_positive, "fn": false_negative,
            "mean_matched_iou": float(np.mean(tp_iou)) if tp_iou else 0.0}


def mean_dice(ground_truth, predictions):
    """Mean Dice per annotator-image: the metric the course grades on.

    Computed on the union of the instances so that it measures pixel overlap and
    nothing about how the mask is split into objects.
    """
    scores = []
    for entry in ground_truth:
        labels, n_pred, pred_areas = predictions.get(entry["stem"], (None, 0, None))
        gt_index = entry["union"]
        pred_total = int(pred_areas.sum()) if n_pred else 0
        if gt_index.size == 0 and pred_total == 0:
            scores.append(1.0)
            continue
        overlap = int((labels[gt_index] > 0).sum()) if n_pred and gt_index.size else 0
        scores.append((2 * overlap + 1e-6) / (gt_index.size + pred_total + 1e-6))
    return float(np.mean(scores)) if scores else 0.0

## 7. Operating point, against the official metric

A detector has three knobs where the U-Net had four: the confidence a candidate needs
to exist (`conf`), how aggressively overlapping candidates are suppressed (NMS `iou`;
`0.0` means any two touching boxes keep only the stronger), and a cap on instances per
image (`max_det`). Predictions are computed once per `iou` at a permissive confidence;
`conf` and `max_det` are applied afterwards, which is exact because NMS never lets a
weaker box suppress a stronger one. Overlapping masks are made disjoint in confidence
order, as the official scorer expects one label per pixel.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image as PILImage

ground_truth = build_ground_truth(by_split["validation"])
print(f"{len(ground_truth)} annotator-image entries, "
      f"{sum(len(e['instances']) for e in ground_truth)} ground-truth filaments")

validation_files = {}
for sample in by_split["validation"]:
    validation_files.setdefault(Path(sample.file_name).stem, sample.image_path)


def predict_candidates(paths, nms_iou):
    """Per file: confidences and RLE masks at native resolution, NMS applied."""
    out = {}
    for stem, path in tqdm(paths.items(), desc=f"predict iou={nms_iou:.1f}", leave=False):
        result = yolo.predict(str(path), imgsz=IMG_SIZE, conf=PREDICT_CONF, iou=nms_iou,
                              max_det=max(SEARCH_MAX_DET), retina_masks=RETINA_MASKS,
                              device=device, verbose=False)[0]
        items = []
        if result.masks is not None:
            masks = result.masks.data.cpu().numpy().astype(np.uint8)
            confs = result.boxes.conf.cpu().numpy()
            for mask, conf in zip(masks, confs):
                if mask.shape != (NATIVE, NATIVE):
                    mask = np.asarray(PILImage.fromarray(mask).resize(
                        (NATIVE, NATIVE), PILImage.Resampling.NEAREST))
                rle = mask_util.encode(np.asfortranarray(mask))
                items.append((float(conf), rle))
        out[stem] = sorted(items, key=lambda item: -item[0])
    return out


def instantiate(candidates, conf_threshold, max_det):
    """Label image per file: disjoint instances painted in confidence order."""
    predictions = {}
    for stem, items in candidates.items():
        kept = [item for item in items if item[0] >= conf_threshold][:max_det]
        labels = np.zeros(NATIVE * NATIVE, dtype=np.int32)
        areas = []
        for label, (conf, rle) in enumerate(kept, start=1):
            pixels = np.flatnonzero(mask_util.decode(rle).ravel())
            pixels = pixels[labels[pixels] == 0]
            if pixels.size < 5:
                continue
            labels[pixels] = len(areas) + 1
            areas.append(pixels.size)
        predictions[stem] = (labels, len(areas), np.array(areas, dtype=np.int64))
    return predictions


started = time.perf_counter()
grid, candidates_by_iou = [], {}
for nms_iou in SEARCH_IOU:
    candidates_by_iou[nms_iou] = predict_candidates(validation_files, nms_iou)
    for conf_threshold in SEARCH_CONF:
        for max_det in SEARCH_MAX_DET:
            predictions = instantiate(candidates_by_iou[nms_iou], conf_threshold, max_det)
            row = {"iou": nms_iou, "conf": conf_threshold, "max_det": max_det,
                   "dice": mean_dice(ground_truth, predictions),
                   **official_pq(ground_truth, predictions)}
            grid.append(row)
    best_here = max((r for r in grid if r["iou"] == nms_iou), key=lambda r: r["pq"])
    print(f"iou={nms_iou:.1f}: best PQ {best_here['pq']:.4f} at conf={best_here['conf']:.2f} "
          f"max_det={best_here['max_det']}  ({time.perf_counter() - started:.0f}s)")

operating_point = max(grid, key=lambda r: (r["pq"], r["dice"]))
predictions = instantiate(candidates_by_iou[operating_point["iou"]],
                          operating_point["conf"], operating_point["max_det"])
denominator = operating_point["tp"] + 0.5 * operating_point["fp"] + 0.5 * operating_point["fn"]
print(f"\nselected: iou={operating_point['iou']:.1f} conf={operating_point['conf']:.2f} "
      f"max_det={operating_point['max_det']} -> PQ {operating_point['pq']:.4f} "
      f"(SQ {operating_point['mean_matched_iou']:.4f} RQ {operating_point['tp'] / denominator:.4f}) "
      f"Dice {operating_point['dice']:.4f}  TP {operating_point['tp']} "
      f"FP {operating_point['fp']} FN {operating_point['fn']}")
save_json({"selected": operating_point, "grid": grid}, METRICS_DIR / "operating_point.json")

recipe = {"experiment_id": EXPERIMENT_ID, "detector": YOLO_MODEL,
          "checkpoint": str(BEST_CHECKPOINT.relative_to(PROJECT_ROOT)),
          "image_size": IMG_SIZE, "conf": operating_point["conf"],
          "iou": operating_point["iou"], "max_det": operating_point["max_det"],
          "retina_masks": RETINA_MASKS,
          "validation_dice": operating_point["dice"], "validation_pq": operating_point["pq"]}
save_json(recipe, METRICS_DIR / "submission_recipe.json")
print("\n" + json.dumps(recipe, indent=2))

figure, axis = plt.subplots(figsize=(7, 4))
for nms_iou in SEARCH_IOU:
    curve = [max((r for r in grid if r["iou"] == nms_iou and r["conf"] == c),
                 key=lambda r: r["pq"]) for c in SEARCH_CONF]
    axis.plot(SEARCH_CONF, [r["pq"] for r in curve], marker="o", label=f"NMS iou {nms_iou:.1f}")
axis.axvline(operating_point["conf"], color="tab:red", linestyle="--",
             label=f"selected conf={operating_point['conf']:.2f}")
axis.set(xlabel="confidence threshold", ylabel="validation PQ (best max_det)")
axis.grid(alpha=0.3); axis.legend(fontsize=8)
figure.tight_layout()
figure.savefig(PLOTS_DIR / "threshold_search.png", dpi=160, bbox_inches="tight")
plt.show()

## 11. Where the error actually is

A single PQ number does not say what to fix next. This cell breaks the error down:
instance recall by filament size, and *why* each false negative (missed completely /
found but too small / split) and each false positive (hallucination / on an annotated
but unmatched filament / merge) happened. These counts are what decided the next
experiment at every step of the project.

In [ ]:

buckets = [(0, 500), (500, 1000), (1000, 2000), (2000, 5000), (10 ** 9,)]
edges = [(0, 500), (500, 1000), (1000, 2000), (2000, 5000), (5000, 10 ** 9)]
found = {b: 0 for b in edges}
total = {b: 0 for b in edges}
for entry in ground_truth:
    labels, n_pred, pred_areas = predictions.get(entry["stem"], (None, 0, np.zeros(0)))
    for index in entry["instances"]:
        bucket = next(b for b in edges if b[0] <= index.size < b[1])
        total[bucket] += 1
        if n_pred:
            counts = np.bincount(labels[index], minlength=n_pred + 1)[1:]
            union = index.size + pred_areas - counts
            if (np.divide(counts, np.maximum(union, 1)) > 0.5).any():
                found[bucket] += 1

print(f"{'filament size (px @2048)':>26} {'total':>6} {'matched':>8} {'recall':>7}")
recall_rows = []
for bucket in edges:
    if not total[bucket]:
        continue
    label = f"{bucket[0]}-{bucket[1]}" if bucket[1] < 10 ** 8 else f">{bucket[0]}"
    recall = found[bucket] / total[bucket]
    print(f"{label:>26} {total[bucket]:>6} {found[bucket]:>8} {recall:>7.2f}")
    recall_rows.append({"range": label, "total": total[bucket],
                        "matched": found[bucket], "recall": recall})
overall = sum(found.values()) / max(sum(total.values()), 1)
print(f"\noverall instance recall: {overall:.3f}")
denominator = (operating_point["tp"] + 0.5 * operating_point["fp"]
               + 0.5 * operating_point["fn"])
sq = operating_point["mean_matched_iou"]
rq = operating_point["tp"] / denominator if denominator else 0.0
print(f"TP {operating_point['tp']}  FP {operating_point['fp']}  FN {operating_point['fn']}  "
      f"SQ {sq:.4f}  RQ {rq:.4f}  (PQ = SQ x RQ = {sq * rq:.4f})")


def error_causes(ground_truth, predictions, iou_threshold=0.5):
    """Why each FN and FP happened, in the operational sense that decides what to
    build next.

    FN: `pure_miss` - not one predicted pixel on the instance; `split` - no single
    prediction reaches the IoU gate but the union of the overlapping ones does;
    `near_miss` - one prediction overlaps but stays at IoU <= 0.5 (its
    predicted/true area ratio is recorded). FP: `hallucination` - the prediction
    touches no ground-truth pixel of that annotator entry; `merge` - it overlaps
    two or more instances and clears the gate against their union; `unmatched` -
    it overlaps annotated filament pixels without matching any single instance.
    """
    fn = {"pure_miss": 0, "near_miss": 0, "split": 0}
    fp = {"hallucination": 0, "merge": 0, "unmatched": 0}
    near_miss_area_ratio = []
    for entry in ground_truth:
        labels, n_pred, pred_areas = predictions.get(
            entry["stem"], (None, 0, np.zeros(0, dtype=np.int64)))
        union_index = entry["union"]
        n_gt = len(entry["instances"])
        if n_pred == 0:
            fn["pure_miss"] += n_gt
            continue
        overlap = (np.stack([np.bincount(labels[ix], minlength=n_pred + 1)[1:]
                             for ix in entry["instances"]])
                   if n_gt else np.zeros((0, n_pred), dtype=np.int64))
        sizes = np.array([ix.size for ix in entry["instances"]], dtype=np.int64)
        iou = np.zeros(overlap.shape)
        if n_gt:
            denom = sizes[:, None] + pred_areas[None, :] - overlap
            np.divide(overlap, np.maximum(denom, 1), out=iou, where=denom > 0)
        hit = iou > iou_threshold
        for i in range(n_gt):
            if hit[i].any():
                continue
            touching = overlap[i] > 0
            if not touching.any():
                fn["pure_miss"] += 1
                continue
            inter = int(overlap[i].sum())
            union = int(sizes[i] + pred_areas[touching].sum() - inter)
            if touching.sum() >= 2 and inter / max(union, 1) > iou_threshold:
                fn["split"] += 1
            else:
                fn["near_miss"] += 1
                best = int(np.argmax(iou[i]))
                near_miss_area_ratio.append(float(pred_areas[best] / max(sizes[i], 1)))
        gt_cover = (np.bincount(labels[union_index], minlength=n_pred + 1)[1:]
                    if union_index.size else np.zeros(n_pred, dtype=np.int64))
        for j in range(n_pred):
            if n_gt and hit[:, j].any():
                continue
            if gt_cover[j] == 0:
                fp["hallucination"] += 1
                continue
            touching = overlap[:, j] > 0 if n_gt else np.zeros(0, bool)
            if touching.sum() >= 2:
                inter = int(overlap[touching, j].sum())
                union = int(sizes[touching].sum() + pred_areas[j] - inter)
                if inter / max(union, 1) > iou_threshold:
                    fp["merge"] += 1
                    continue
            fp["unmatched"] += 1
    ratio = (float(np.median(near_miss_area_ratio))
             if near_miss_area_ratio else float("nan"))
    return fn, fp, ratio


fn_causes, fp_causes, near_miss_ratio = error_causes(ground_truth, predictions)
print(f"FN causes: {fn_causes}  (near-miss median predicted/true area {near_miss_ratio:.2f})")
print(f"FP causes: {fp_causes}")
save_json({"recall_by_size": recall_rows, "overall_instance_recall": overall,
           "sq": sq, "rq": rq,
           "fn_causes": fn_causes, "fp_causes": fp_causes,
           "near_miss_median_area_ratio": near_miss_ratio,
           "operating_point": operating_point}, METRICS_DIR / "error_analysis.json")

## 11b. The distributions the judging committee looks at

The organisers' self-evaluation notebook plots, besides PQ, the distribution of IoU
and Dice over every overlapping ground-truth/prediction pair and the **many-to-one /
one-to-many relations** — how many partners each instance touches. The judging
committee reads these alongside the score, so they are reproduced here with the same
conventions: pairs with zero overlap are dropped, and a "hit" is any overlap at all
(not the IoU > 0.5 gate of the metric).


In [ ]:
iou_pairs, dice_pairs = [], []
gt_degrees, pred_degrees = [], []
for entry in ground_truth:
    labels, n_pred, pred_areas = predictions.get(
        entry["stem"], (None, 0, np.zeros(0, dtype=np.int64)))
    sizes = np.array([index.size for index in entry["instances"]], dtype=np.int64)
    n_gt = sizes.size
    if n_gt == 0:
        pred_degrees.extend([0] * n_pred)
        continue
    if n_pred == 0:
        gt_degrees.extend([0] * n_gt)
        continue
    inter = np.stack([np.bincount(labels[index], minlength=n_pred + 1)[1:]
                      for index in entry["instances"]])
    total = sizes[:, None] + pred_areas[None, :]
    union = total - inter
    iou = np.zeros(inter.shape)
    np.divide(inter, np.maximum(union, 1), out=iou, where=union > 0)
    dice = np.zeros(inter.shape)
    np.divide(2 * inter, np.maximum(total, 1), out=dice, where=union > 0)
    hit = inter > 0
    iou_pairs.extend(iou[hit].tolist())
    dice_pairs.extend(dice[hit].tolist())
    gt_degrees.extend(hit.sum(axis=1).tolist())
    pred_degrees.extend(hit.sum(axis=0).tolist())

from collections import Counter as _Counter
gt_deg, pred_deg = _Counter(gt_degrees), _Counter(pred_degrees)

figure, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, scores, name in ((axes[0], iou_pairs, "IoU"), (axes[1], dice_pairs, "Dice")):
    scores = np.asarray(scores)
    counts, edges = np.histogram(scores, bins=np.linspace(0, 1, 51))
    axis.bar((edges[:-1] + edges[1:]) / 2, counts / max(counts.sum(), 1), width=0.02,
             edgecolor="black", alpha=0.7, color="navy")
    axis.axvline(scores.mean(), color="crimson", linestyle="--",
                 label=f"mean = {scores.mean():.3f}")
    axis.set(title=f"{name} distribution (overlapping pairs)", xlabel=name,
             ylabel="probability")
    axis.grid(axis="y", linestyle="--", alpha=0.5); axis.legend(fontsize=8)

axis = axes[2]
for degree, count in sorted(gt_deg.items()):
    axis.bar(degree + 0.5, count, width=0.8, edgecolor="black", alpha=0.7,
             color="crimson" if degree == 0 else "forestgreen")
for degree, count in sorted(pred_deg.items()):
    axis.bar(-(degree + 0.5), count, width=0.8, edgecolor="black", alpha=0.7,
             color="darkorange" if degree == 0 else "steelblue")
positions = ([d + 0.5 for d in sorted(gt_deg)]
             + [-(d + 0.5) for d in sorted(pred_deg)])
axis.set_xticks(positions)
axis.set_xticklabels([f"1:{int(round(p - 0.5))}" if p > 0 else
                      f"{int(round(-p - 0.5))}:1" for p in positions],
                     rotation=45, fontsize=8)
axis.axvline(0, color="black", linewidth=0.8)
axis.set(title="many-to-one (left) / one-to-many (right)", ylabel="frequency")
axis.grid(axis="y", linestyle="--", alpha=0.5)
figure.tight_layout()
figure.savefig(PLOTS_DIR / "official_distributions.png", dpi=160, bbox_inches="tight")
plt.show()

gt_hit = sum(c for d, c in gt_deg.items() if d > 0)
pred_hit = sum(c for d, c in pred_deg.items() if d > 0)
print(f"GT instances:   {gt_hit} touched by >=1 prediction, {gt_deg.get(0, 0)} untouched")
print(f"predictions:    {pred_hit} touching >=1 GT instance, {pred_deg.get(0, 0)} touching none")
print(f"one-to-many GT (1:n, n>=2): {sum(c for d, c in gt_deg.items() if d >= 2)}"
      f" | many-to-one predictions (n:1, n>=2): {sum(c for d, c in pred_deg.items() if d >= 2)}")
save_json({"gt_degree_counts": {str(k): v for k, v in sorted(gt_deg.items())},
           "pred_degree_counts": {str(k): v for k, v in sorted(pred_deg.items())},
           "mean_pair_iou": float(np.mean(iou_pairs)) if iou_pairs else 0.0,
           "mean_pair_dice": float(np.mean(dice_pairs)) if dice_pairs else 0.0},
          METRICS_DIR / "official_distributions.json")

In [ ]:
summary = {"experiment_id": EXPERIMENT_ID, "training": training_summary,
           "operating_point": recipe}
save_json(summary, EXPERIMENT_DIR / "summary.json")
print(f"Next: open notebooks/yolo/02_inference.ipynb with EXPERIMENT_ID = \"{EXPERIMENT_ID}\".")